In [1]:
from langchain_community.document_loaders import CSVLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

from langchain_classic.chains import RetrievalQA
from langchain_huggingface import HuggingFacePipeline

J:\Anaconda py\envs\langchain_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

import nltk                    
nltk.download('punkt', quiet=True)   


import os, math, random, textwrap  
import numpy as np                  
import pandas as pd                 
from typing import List, Dict      


try:
    
    from langchain_community.document_loaders import CSVLoader, TextLoader, WikipediaLoader
    from langchain_community.vectorstores import FAISS
    from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
except Exception:

    from langchain.document_loaders import CSVLoader, TextLoader
    try:
        from langchain.document_loaders import WikipediaLoader
    except Exception:
        WikipediaLoader = None
    from langchain.vectorstores import FAISS
    try:
        from langchain.embeddings import HuggingFaceEmbeddings
    except Exception:
        pass
    try:
        from langchain.llms import HuggingFacePipeline
    except Exception:
        pass


from langchain_classic.chains import RetrievalQA                              


from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSeq2SeqLM, pipeline

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction


SEED = 42
random.seed(SEED); np.random.seed(SEED)

print("Setup complete.")


Setup complete.


In [3]:
!pip install wikipedia
DATA_PATH = "wiki_sample.csv"  

docs = []   

if os.path.exists(DATA_PATH):
    
    loader = CSVLoader(DATA_PATH)         
    docs = loader.load()                   
    source_info = f"Loaded {len(docs)} rows from CSV."
elif 'WikipediaLoader' in globals() and WikipediaLoader is not None:

    loader = WikipediaLoader(query="Artificial intelligence", load_max_docs=3)
    docs = loader.load()
    source_info = f"Loaded {len(docs)} wiki pages via WikipediaLoader."
else:
   
    from langchain.schema import Document
    sample_texts = [
        "Machine learning is a subset of artificial intelligence focused on algorithms that learn patterns from data.",
        "Deep learning uses neural networks with many layers to model complex representations and achieve state-of-the-art results."
    ]
    docs = [Document(page_content=t, metadata={"source":"inline"}) for t in sample_texts]
    source_info = "Using tiny inline sample (offline fallback)."

print("Source:", source_info)
print("First doc snippet:", docs[0].page_content[:120], "...")


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,          
    chunk_overlap=100,       
    separators=["\n\n", "\n", " ", ""],  
)

splits = text_splitter.split_documents(docs)  # List[Document] of chunks
print(f"Created {len(splits)} chunks.")


for i, d in enumerate(splits[:5]):
    print(f"--- Chunk {i+1} (len={len(d.page_content)}) ---")
    print(textwrap.shorten(d.page_content, width=200, placeholder=" ..."))



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Source: Loaded 3 wiki pages via WikipediaLoader.
First doc snippet: Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human ...
Created 35 chunks.
--- Chunk 1 (len=463) ---
Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and ...
--- Chunk 2 (len=498) ---
High-profile applications of AI include advanced web search engines (e.g., Google Search); recommendation systems (used by YouTube, Amazon, and Netflix); virtual assistants (e.g., Google ...
--- Chunk 3 (len=236) ---
many AI applications are not perceived as such: "A lot of cutting-edge AI has filtered into general applications, often without being called AI because once something becomes useful enough and ...
--- Chunk 4 (len=498) ---
Various subfields of AI research are centered around particular goals and the use of particular

In [6]:


EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2" 


embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)


vector_store = FAISS.from_documents(splits, embeddings)


query = "What is machine learning?"
results = vector_store.similarity_search(query, k=2) 
print("\nQuery:", query)
for i, r in enumerate(results, start=1):
    print(f"#{i} (len={len(r.page_content)}):", textwrap.shorten(r.page_content, width=180, placeholder=" ..."))


Loading weights: 100%|█████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 1079.78it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Query: What is machine learning?
#1 (len=489): Artificial intelligence is the capability of the computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem- ...
#2 (len=463): Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem- ...


In [25]:
# from transformers import pipeline
# from langchain_huggingface import HuggingFacePipeline
# GEN_MODEL = "google/flan-t5-small"

# hf_pipe = pipeline(task="text-generation", model=GEN_MODEL)

# llm = HuggingFacePipeline(pipeline=hf_pipe)


# prompt = "Explain artificial intelligence in one sentence."
# print("Prompt:", prompt)
# output = llm(prompt)
# print("Output:", output)

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_classic.chains import RetrievalQA


GEN_MODEL = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(GEN_MODEL)


hf_pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)


llm = HuggingFacePipeline(pipeline=hf_pipe)


Loading weights: 100%|█████████████████████████████████████████████████████████████| 190/190 [00:00<00:00, 1561.32it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', '

In [29]:

retriever = vector_store.as_retriever(search_kwargs={"k": 3})  

qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff"    )

sample_queries = [
    "What is deep learning?",
    "Define machine learning.",
    "How do neural networks learn?",
    "What is the role of embeddings?",
    "Explain the purpose of a vector database."
]

print("🔗 RAG Responses:")
for q in sample_queries:
    ans = qa.invoke(q)         
    print("\nQ:", q)
    print("A:", ans["result"])


🔗 RAG Responses:


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: What is deep learning?
A: Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

Artificial intelligence is the capability of the  computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. Artificial intelligence has been used in applications throughout industry and academia. Within the field of Artificial Intelligence, there are multiple subfields. The subfield of Machine learning has been used for various scientific and commercial purposes including language

Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in computer science that develops and studies methods and software that ena

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: Define machine learning.
A: Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

Artificial intelligence is the capability of the  computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. Artificial intelligence has been used in applications throughout industry and academia. Within the field of Artificial Intelligence, there are multiple subfields. The subfield of Machine learning has been used for various scientific and commercial purposes including language

Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in computer science that develops and studies methods and software that e

...


Q: How do neural networks learn?
A: Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

==== Quantum computing ====

Research and development of quantum computers has been performed with machine learning algorithms. For example, there is a prototype, photonic, quantum memristive device for neuromorphic computers (NC)/artificial neural networks and NC-using quantum materials with some variety of potential neuromorphic computing-related applications. The use of quantum machine learning  for quantum simulators has been proposed for solving physics and chemistry problems.

Various subfields of AI research are centered around particular goals and the use of particular tools. The traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, and perception, as well as support for robotics. To reach these goals, AI res

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: What is the role of embeddings?
A: Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

of Machine learning has been used for various scientific and commercial purposes including language translation, image recognition, decision-making, credit scoring, and e-commerce. In recent years, there have been massive advancements in the field of generative artificial intelligence, which uses generative models to produce text, images, videos or other forms of data. This article describes applications of AI in different sectors.

=== Knowledge representation ===

Knowledge representation and knowledge engineering allow AI programs to answer questions i

automatically lead to increases in revenue or actual productivity. Referring to "AI generated work content that masquerades as good work, but lacks the substance to meaningfully advance a given task" the article coins the term workslo

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: Explain the purpose of a vector database.
A: Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

=== Knowledge representation ===

Knowledge representation and knowledge engineering allow AI programs to answer questions i

Artificial intelligence is the capability of the  computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. Artificial intelligence has been used in applications throughout industry and academia. Within the field of Artificial Intelligence, there are multiple subfields. The subfield of Machine learning has been used for various scientific and commercial purposes including language

== Agriculture ==

In agriculture, AI has been proposed as a way for farmers to identify areas that need irrigation, fertilization, or pesticide treatments to increase yi

In [31]:

def rebuild_pipeline(chunk_size=700, chunk_overlap=120, k=4, embed_model=EMBED_MODEL):
    """Rebuilds the RAG pipeline with different chunking, k, or embedding model."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", " ", ""]
    )
    chunks = splitter.split_documents(docs)
    _emb = HuggingFaceEmbeddings(model_name=embed_model)
    _vs = FAISS.from_documents(chunks, _emb)
    _retr = _vs.as_retriever(search_kwargs={"k": k})
    _qa = RetrievalQA.from_chain_type(llm=llm, retriever=_retr, chain_type="stuff")
    return _qa, _vs, chunks

qa2, vs2, chunks2 = rebuild_pipeline(chunk_size=700, chunk_overlap=120, k=4)

test_queries = ["What is deep learning?", "What is machine learning?"]

print("After refinement:")
for q in test_queries:
    print("\nQ:", q)
    print("A:", qa2.invoke(q)["result"])

print(f"\nOld #chunks: {len(splits)} | New #chunks: {len(chunks2)}")


Loading weights: 100%|█████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 1907.15it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


After refinement:

Q: What is deep learning?


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.

Various subfields of AI research are centered around particular goals and the use of particular tools. The traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, and perception, as well as support for robotics. To reach these goals, AI researchers have adapted and integrated a wide range of tech

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.

The following outline is provided as an overview of and topical guide to artificial intelligence:
Artificial intelligence (AI) is intelligence exhibited by machines or software. It is also the name of the scientific field which studies how to create computers and computer software that are capable of intelligent behavior.


== AI algorithms and techniques ==

Artifi

In [33]:

smooth = SmoothingFunction().method1  

eval_queries = [
    "What is deep learning?",
    "Define machine learning.",
    "How do neural networks learn?",
    "What is the role of embeddings?",
    "Explain the purpose of a vector database."
]

references = {
    "What is deep learning?": "Deep learning is a subset of machine learning that uses multi-layer neural networks to learn hierarchical representations.",
    "Define machine learning.": "Machine learning is a field of AI where algorithms learn patterns from data to make predictions or decisions without explicit rules.",
    "How do neural networks learn?": "Neural networks learn by adjusting weights via backpropagation and gradient descent to minimize a loss function on training data.",
    "What is the role of embeddings?": "Embeddings map items like words or passages to dense vectors so that semantic similarity corresponds to geometric proximity.",
    "Explain the purpose of a vector database.": "A vector database stores embeddings and supports fast similarity search to retrieve semantically relevant items."
}

rows = []
for q in eval_queries:
    generated = qa.invoke(q)["result"]
    ref = references[q]
    bleu = sentence_bleu([ref.split()], generated.split(), smoothing_function=smooth) 
    rows.append({"query": q, "generated": generated, "reference": ref, "bleu": bleu})

bleu_df = pd.DataFrame(rows)
bleu_df


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

,query,generated,reference,bleu
0,What is deep learning?,Use the following pieces of context to answer ...,Deep learning is a subset of machine learning ...,0.002401
1,Define machine learning.,Use the following pieces of context to answer ...,Machine learning is a field of AI where algori...,0.014890
2,How do neural networks learn?,Use the following pieces of context to answer ...,Neural networks learn by adjusting weights via...,0.001262
3,What is the role of embeddings?,Use the following pieces of context to answer ...,Embeddings map items like words or passages to...,0.001441
4,Explain the purpose of a vector database.,Use the following pieces of context to answer ...,A vector database stores embeddings and suppor...,0.001319


In [35]:

OPT_EMBED_MODEL = "sentence-transformers/all-mpnet-base-v2"  

qa_opt, vs_opt, chunks_opt = rebuild_pipeline(
    chunk_size=700, chunk_overlap=120, k=5, embed_model=OPT_EMBED_MODEL
)

opt_queries = [
    "What is deep learning?",
    "Define machine learning.",
    "What is the role of embeddings?"
]

rows2 = []
for q in opt_queries:
    gen = qa_opt.invoke(q)["result"]
    ref = references[q]
    bleu = sentence_bleu([ref.split()], gen.split(), smoothing_function=smooth)
    rows2.append({"query": q, "generated_opt": gen, "reference": ref, "bleu_opt": bleu})

bleu_opt_df = pd.DataFrame(rows2)
bleu_opt_df


J:\Anaconda py\envs\langchain_env\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\hp\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████████████████████████████████████████████████████████| 199/199 [00:

,query,generated_opt,reference,bleu_opt
0,What is deep learning?,Use the following pieces of context to answer ...,Deep learning is a subset of machine learning ...,0.001235
1,Define machine learning.,Use the following pieces of context to answer ...,Machine learning is a field of AI where algori...,0.007660
2,What is the role of embeddings?,Use the following pieces of context to answer ...,Embeddings map items like words or passages to...,0.000756
